In [1]:

# 01. LOAD CLEANED DATASETS

import pandas as pd
import os


print("                 LOADING CLEANED DATASETS")

# ------------------------------------------------
# 2. Load Cleaned Datasets
# ------------------------------------------------

customers = pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\customers_clean.csv")

orders = pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\orders_clean.csv")

order_items = pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\order_items_clean.csv")

payments =  pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\payments_clean.csv")

reviews =  pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\reviews_clean.csv")

products =  pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\products_clean.csv")

sellers =  pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\sellers_clean.csv")

geolocation =  pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\geolocation_clean.csv")

category_translation =  pd.read_csv(r"C:\Users\Rajan\OneDrive\Desktop\ecommerce-business-intelligence\Data\cleaned\category_translation_clean.csv")


# 3. Confirm Loading

print("\nAll cleaned datasets loaded successfully.")

print("\nDataset Shapes:")

print(f"Customers              : {customers.shape}")
print(f"Orders                 : {orders.shape}")
print(f"Order Items            : {order_items.shape}")
print(f"Payments               : {payments.shape}")
print(f"Reviews                : {reviews.shape}")
print(f"Products               : {products.shape}")
print(f"Sellers                : {sellers.shape}")
print(f"Geolocation            : {geolocation.shape}")
print(f"Category Translation   : {category_translation.shape}")

print("CLEANED DATA LOADING COMPLETED")

                 LOADING CLEANED DATASETS

All cleaned datasets loaded successfully.

Dataset Shapes:
Customers              : (99441, 5)
Orders                 : (99441, 8)
Order Items            : (112650, 7)
Payments               : (103886, 5)
Reviews                : (99224, 7)
Products               : (32951, 9)
Sellers                : (3095, 4)
Geolocation            : (738332, 5)
Category Translation   : (71, 2)
CLEANED DATA LOADING COMPLETED


In [2]:
#Create dataset
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

In [4]:
# ================================================================
# 05. DATA MODEL
# STEP 1 — PRIMARY KEY & FOREIGN KEY INSPECTION
# ================================================================

print("=" * 80)
print("             PRIMARY KEY & FOREIGN KEY INSPECTION")
print("=" * 80)


# ------------------------------------------------
# Dataset Columns
# ------------------------------------------------

for name, df in datasets.items():

    print("\n" + "-" * 80)
    print(f"{name}")
    print("-" * 80)

    print(f"Rows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")

    print("\nColumns:")
    
    for column in df.columns:
        print(f"  • {column}")


print("\n" + "=" * 80)
print("KEY INSPECTION COMPLETED")
print("=" * 80)

             PRIMARY KEY & FOREIGN KEY INSPECTION

--------------------------------------------------------------------------------
Customers
--------------------------------------------------------------------------------
Rows    : 99,441
Columns : 5

Columns:
  • customer_id
  • customer_unique_id
  • customer_zip_code_prefix
  • customer_city
  • customer_state

--------------------------------------------------------------------------------
Orders
--------------------------------------------------------------------------------
Rows    : 99,441
Columns : 8

Columns:
  • order_id
  • customer_id
  • order_status
  • order_purchase_timestamp
  • order_approved_at
  • order_delivered_carrier_date
  • order_delivered_customer_date
  • order_estimated_delivery_date

--------------------------------------------------------------------------------
Order Items
--------------------------------------------------------------------------------
Rows    : 112,650
Columns : 7

Columns:
  • order_i

In [5]:
# 05. DATA MODEL
# STEP 2 — PRIMARY KEY VALIDATION

print("                    PRIMARY KEY VALIDATION")



# Candidate Primary Keys


primary_key_candidates = {
    "Customers": ["customer_id", "customer_unique_id"],
    "Orders": ["order_id"],
    "Order Items": ["order_id", "order_item_id"],
    "Payments": ["order_id", "payment_sequential"],
    "Reviews": ["review_id"],
    "Products": ["product_id"],
    "Sellers": ["seller_id"],
    "Geolocation": ["geolocation_zip_code_prefix"],
    "Category Translation": ["product_category_name"]
}



# Validate Keys

for table_name, keys in primary_key_candidates.items():

    df = datasets[table_name]

    print("\n" + "-" * 80)
    print(table_name)
    print("-" * 80)

    for key in keys:

        if key not in df.columns:
            print(f"\n{key} → COLUMN NOT FOUND")
            continue

        null_count = df[key].isnull().sum()
        duplicate_count = df[key].duplicated().sum()
        unique_count = df[key].nunique()

        print(f"\n{key}")
        print(f"  Total Rows     : {len(df):,}")
        print(f"  Unique Values  : {unique_count:,}")
        print(f"  Null Values    : {null_count:,}")
        print(f"  Duplicates     : {duplicate_count:,}")


print("PRIMARY KEY VALIDATION COMPLETED")


                    PRIMARY KEY VALIDATION

--------------------------------------------------------------------------------
Customers
--------------------------------------------------------------------------------

customer_id
  Total Rows     : 99,441
  Unique Values  : 99,441
  Null Values    : 0
  Duplicates     : 0

customer_unique_id
  Total Rows     : 99,441
  Unique Values  : 96,096
  Null Values    : 0
  Duplicates     : 3,345

--------------------------------------------------------------------------------
Orders
--------------------------------------------------------------------------------

order_id
  Total Rows     : 99,441
  Unique Values  : 99,441
  Null Values    : 0
  Duplicates     : 0

--------------------------------------------------------------------------------
Order Items
--------------------------------------------------------------------------------

order_id
  Total Rows     : 112,650
  Unique Values  : 98,666
  Null Values    : 0
  Duplicates     : 13,984


In [6]:

# ================================================================
# 05. DATA MODEL
# STEP 3 — COMPOSITE KEY VALIDATION
# ================================================================

print("=" * 80)
print("                  COMPOSITE KEY VALIDATION")
print("=" * 80)


# ------------------------------------------------
# 1. Order Items Composite Key
# ------------------------------------------------

order_items_key_duplicates = (
    order_items
    .duplicated(subset=["order_id", "order_item_id"])
    .sum()
)

order_items_key_unique = (
    order_items[["order_id", "order_item_id"]]
    .drop_duplicates()
    .shape[0]
)


print("\nORDER ITEMS")
print("-" * 80)

print(f"Total Rows                    : {len(order_items):,}")
print(f"Unique (order_id, order_item_id): {order_items_key_unique:,}")
print(f"Composite Key Duplicates      : {order_items_key_duplicates:,}")


# ------------------------------------------------
# 2. Payments Composite Key
# ------------------------------------------------

payments_key_duplicates = (
    payments
    .duplicated(subset=["order_id", "payment_sequential"])
    .sum()
)

payments_key_unique = (
    payments[["order_id", "payment_sequential"]]
    .drop_duplicates()
    .shape[0]
)


print("\nPAYMENTS")
print("-" * 80)

print(f"Total Rows                         : {len(payments):,}")
print(f"Unique (order_id, payment_sequential): {payments_key_unique:,}")
print(f"Composite Key Duplicates           : {payments_key_duplicates:,}")


print("\n" + "=" * 80)
print("COMPOSITE KEY VALIDATION COMPLETED")
print("=" * 80)

                  COMPOSITE KEY VALIDATION

ORDER ITEMS
--------------------------------------------------------------------------------
Total Rows                    : 112,650
Unique (order_id, order_item_id): 112,650
Composite Key Duplicates      : 0

PAYMENTS
--------------------------------------------------------------------------------
Total Rows                         : 103,886
Unique (order_id, payment_sequential): 103,886
Composite Key Duplicates           : 0

COMPOSITE KEY VALIDATION COMPLETED


In [7]:
# ================================================================
# 05. DATA MODEL
# STEP 4 — FOREIGN KEY VALIDATION
# ================================================================

print("=" * 80)
print("                    FOREIGN KEY VALIDATION")
print("=" * 80)


# ------------------------------------------------
# Function to Validate Foreign Key
# ------------------------------------------------

def validate_foreign_key(
    child_df,
    child_column,
    parent_df,
    parent_column,
    relationship_name
):
    
    child_values = set(child_df[child_column].dropna().unique())
    parent_values = set(parent_df[parent_column].dropna().unique())

    unmatched_values = child_values - parent_values

    print(f"\n{relationship_name}")
    print("-" * 80)

    print(f"Child Column       : {child_column}")
    print(f"Parent Column      : {parent_column}")
    print(f"Child Unique IDs   : {len(child_values):,}")
    print(f"Parent Unique IDs  : {len(parent_values):,}")
    print(f"Unmatched IDs      : {len(unmatched_values):,}")

    if len(unmatched_values) == 0:
        print("Status             : PASS")
    else:
        print("Status             : CHECK REQUIRED")

    return unmatched_values


# ------------------------------------------------
# 1. Orders → Customers
# ------------------------------------------------

orders_customer_unmatched = validate_foreign_key(
    orders,
    "customer_id",
    customers,
    "customer_id",
    "Orders.customer_id → Customers.customer_id"
)


# ------------------------------------------------
# 2. Order Items → Orders
# ------------------------------------------------

order_items_order_unmatched = validate_foreign_key(
    order_items,
    "order_id",
    orders,
    "order_id",
    "Order Items.order_id → Orders.order_id"
)


# ------------------------------------------------
# 3. Order Items → Products
# ------------------------------------------------

order_items_product_unmatched = validate_foreign_key(
    order_items,
    "product_id",
    products,
    "product_id",
    "Order Items.product_id → Products.product_id"
)


# ------------------------------------------------
# 4. Order Items → Sellers
# ------------------------------------------------

order_items_seller_unmatched = validate_foreign_key(
    order_items,
    "seller_id",
    sellers,
    "seller_id",
    "Order Items.seller_id → Sellers.seller_id"
)


# ------------------------------------------------
# 5. Payments → Orders
# ------------------------------------------------

payments_order_unmatched = validate_foreign_key(
    payments,
    "order_id",
    orders,
    "order_id",
    "Payments.order_id → Orders.order_id"
)


# ------------------------------------------------
# 6. Reviews → Orders
# ------------------------------------------------

reviews_order_unmatched = validate_foreign_key(
    reviews,
    "order_id",
    orders,
    "order_id",
    "Reviews.order_id → Orders.order_id"
)


print("\n" + "=" * 80)
print("FOREIGN KEY VALIDATION COMPLETED")
print("=" * 80)


                    FOREIGN KEY VALIDATION

Orders.customer_id → Customers.customer_id
--------------------------------------------------------------------------------
Child Column       : customer_id
Parent Column      : customer_id
Child Unique IDs   : 99,441
Parent Unique IDs  : 99,441
Unmatched IDs      : 0
Status             : PASS

Order Items.order_id → Orders.order_id
--------------------------------------------------------------------------------
Child Column       : order_id
Parent Column      : order_id
Child Unique IDs   : 98,666
Parent Unique IDs  : 99,441
Unmatched IDs      : 0
Status             : PASS

Order Items.product_id → Products.product_id
--------------------------------------------------------------------------------
Child Column       : product_id
Parent Column      : product_id
Child Unique IDs   : 32,951
Parent Unique IDs  : 32,951
Unmatched IDs      : 0
Status             : PASS

Order Items.seller_id → Sellers.seller_id
------------------------------------

In [8]:
# ================================================================
# 05. DATA MODEL
# STEP 5 — REVIEWS COMPOSITE KEY VALIDATION
# ================================================================

print("=" * 80)
print("               REVIEWS COMPOSITE KEY VALIDATION")
print("=" * 80)


# ------------------------------------------------
# 1. Validate review_id + order_id
# ------------------------------------------------

review_key_duplicates = (
    reviews
    .duplicated(subset=["review_id", "order_id"])
    .sum()
)

review_key_unique = (
    reviews[["review_id", "order_id"]]
    .drop_duplicates()
    .shape[0]
)


print("\nREVIEWS")
print("-" * 80)

print(f"Total Rows                         : {len(reviews):,}")
print(f"Unique (review_id, order_id)      : {review_key_unique:,}")
print(f"Composite Key Duplicates          : {review_key_duplicates:,}")


# ------------------------------------------------
# 2. Check duplicate review_id examples
# ------------------------------------------------

duplicate_review_ids = (
    reviews[
        reviews["review_id"].duplicated(keep=False)
    ]
    .sort_values("review_id")
)

print("\nDuplicate review_id records:")
print("-" * 80)

print(
    duplicate_review_ids[
        ["review_id", "order_id", "review_score"]
    ].head(10)
)


print("\n" + "=" * 80)
print("REVIEWS KEY VALIDATION COMPLETED")
print("=" * 80)

               REVIEWS COMPOSITE KEY VALIDATION

REVIEWS
--------------------------------------------------------------------------------
Total Rows                         : 99,224
Unique (review_id, order_id)      : 99,224
Composite Key Duplicates          : 0

Duplicate review_id records:
--------------------------------------------------------------------------------
                              review_id                          order_id  \
46678  00130cbe1f9d422698c812ed8ded1919  dfcdfc43867d1c1381bfaf62d6b9c195   
29841  00130cbe1f9d422698c812ed8ded1919  04a28263e085d399c97ae49e0b477efa   
90677  0115633a9c298b6a98bcbe4eee75345f  78a4201f58af3463bdab842eea4bc801   
63193  0115633a9c298b6a98bcbe4eee75345f  0c9850b2c179c1ef60d2855e2751d1fa   
92876  0174caf0ee5964646040cd94e15ac95e  f93a732712407c02dce5dd5088d0f47b   
57280  0174caf0ee5964646040cd94e15ac95e  74db91e33b4e1fd865356c89a61abf1f   
54832  017808d29fd1f942d97e50184dfb4c13  8daaa9e99d60fbba579cc1c3e3bfae01   
99167  017

In [9]:
# ================================================================
# 05. DATA MODEL
# STEP 6 — CATEGORY TRANSLATION VALIDATION
# ================================================================

print("=" * 80)
print("             CATEGORY TRANSLATION VALIDATION")
print("=" * 80)


# ------------------------------------------------
# 1. Get Unique Product Categories
# ------------------------------------------------

product_categories = set(
    products["product_category_name"]
    .dropna()
    .unique()
)

translation_categories = set(
    category_translation["product_category_name"]
    .dropna()
    .unique()
)


# ------------------------------------------------
# 2. Find Unmatched Categories
# ------------------------------------------------

products_without_translation = (
    product_categories - translation_categories
)

translation_without_products = (
    translation_categories - product_categories
)


# ------------------------------------------------
# 3. Display Results
# ------------------------------------------------

print("\nPRODUCT CATEGORIES")
print("-" * 80)

print(f"Unique categories in Products              : {len(product_categories):,}")
print(f"Unique categories in Category Translation  : {len(translation_categories):,}")


print("\nPRODUCT CATEGORIES WITHOUT TRANSLATION")
print("-" * 80)

print(
    f"Unmatched product categories : "
    f"{len(products_without_translation):,}"
)

if products_without_translation:
    print("\nCategories without translation:")
    for category in sorted(products_without_translation):
        print(f"  • {category}")


print("\nTRANSLATION CATEGORIES WITHOUT PRODUCTS")
print("-" * 80)

print(
    f"Unmatched translation categories : "
    f"{len(translation_without_products):,}"
)

if translation_without_products:
    print("\nTranslation categories without products:")
    for category in sorted(translation_without_products):
        print(f"  • {category}")


# ------------------------------------------------
# 4. Overall Status
# ------------------------------------------------

if (
    len(products_without_translation) == 0
    and len(translation_without_products) == 0
):
    status = "PASS"
else:
    status = "CHECK REQUIRED"


print("\n" + "=" * 80)
print(f"CATEGORY TRANSLATION VALIDATION STATUS : {status}")
print("=" * 80)


             CATEGORY TRANSLATION VALIDATION

PRODUCT CATEGORIES
--------------------------------------------------------------------------------
Unique categories in Products              : 74
Unique categories in Category Translation  : 71

PRODUCT CATEGORIES WITHOUT TRANSLATION
--------------------------------------------------------------------------------
Unmatched product categories : 3

Categories without translation:
  • pc_gamer
  • portateis_cozinha_e_preparadores_de_alimentos
  • unknown

TRANSLATION CATEGORIES WITHOUT PRODUCTS
--------------------------------------------------------------------------------
Unmatched translation categories : 0

CATEGORY TRANSLATION VALIDATION STATUS : CHECK REQUIRED


In [10]:
# ================================================================
# 05. DATA MODEL
# STEP 6.1 — UNMATCHED CATEGORY IMPACT ANALYSIS
# ================================================================

print("=" * 80)
print("             UNMATCHED CATEGORY IMPACT ANALYSIS")
print("=" * 80)


# ------------------------------------------------
# 1. Unmatched Categories
# ------------------------------------------------

unmatched_categories = [
    "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos",
    "unknown"
]


# ------------------------------------------------
# 2. Count Products in Each Category
# ------------------------------------------------

category_impact = (
    products[
        products["product_category_name"].isin(unmatched_categories)
    ]
    .groupby("product_category_name")
    .size()
    .reset_index(name="product_count")
)


# ------------------------------------------------
# 3. Calculate Percentage
# ------------------------------------------------

total_products = len(products)

category_impact["percentage_of_products"] = (
    category_impact["product_count"]
    / total_products
    * 100
)


# ------------------------------------------------
# 4. Display Results
# ------------------------------------------------

print("\nUnmatched Category Impact:")
print("-" * 80)

print(
    category_impact.to_string(index=False)
)


# ------------------------------------------------
# 5. Total Impact
# ------------------------------------------------

total_unmatched_products = category_impact["product_count"].sum()

total_unmatched_percentage = (
    total_unmatched_products
    / total_products
    * 100
)


print("\n" + "-" * 80)

print(
    f"Total Products in Unmatched Categories : "
    f"{total_unmatched_products:,}"
)

print(
    f"Percentage of Product Catalog           : "
    f"{total_unmatched_percentage:.2f}%"
)


print("\n" + "=" * 80)
print("UNMATCHED CATEGORY IMPACT ANALYSIS COMPLETED")
print("=" * 80)

             UNMATCHED CATEGORY IMPACT ANALYSIS

Unmatched Category Impact:
--------------------------------------------------------------------------------
                        product_category_name  product_count  percentage_of_products
                                     pc_gamer              3                0.009104
portateis_cozinha_e_preparadores_de_alimentos             10                0.030348
                                      unknown            610                1.851234

--------------------------------------------------------------------------------
Total Products in Unmatched Categories : 623
Percentage of Product Catalog           : 1.89%

UNMATCHED CATEGORY IMPACT ANALYSIS COMPLETED
